# CDMA Module 4 Full Run on Kaggle

This notebook does the following:
1. Clones your GitHub code repository
2. Downloads the features archive from Google Drive
3. Places data into the expected project structure
4. Runs Module 4 (Chapter 7 end-to-end CDMA) for selected conditions across all 5 folds
5. Collects pooled metrics into one summary table

Important: enable **Internet** in Kaggle notebook settings before running.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import zipfile
import re

# --------- User configuration ---------
REPO_URL = "https://github.com/ZahirAhmadChaudhry/CDMA_from_Sctrach.git"
REPO_BRANCH = "main"
PROJECT_DIR = Path("/kaggle/working/CDMA_from_Sctrach")

GDRIVE_FILE_ID = "1LJenZ-VXktBbroTI3btVSkRRq-glSWCb"
ARCHIVE_PATH = Path("/kaggle/working/androids_features.zip")
EXTRACT_DIR = Path("/kaggle/working/androids_extracted")

# Module 4 conditions (set a subset while iterating, then expand as needed)
CONDITIONS = [
    "ba1_rt",
    "ba1_it",
    "itmla_rt",
    "itmla_it",
    "ba2_rt",
    "ba2_it",
    "ba3_rt",
    "ba3_it",
    "ctga_rt",
    "ctga_it",
    "ba4",
    "ba5",
    "full_cdma",
]
REPS = [1]  # change to list(range(1, 11)) for 10 repetitions
EPOCHS = 300

print("Configured conditions:", CONDITIONS)
print("Configured reps:", REPS)
print("Configured epochs:", EPOCHS)

In [ ]:
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

subprocess.run([
    "git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_DIR)
], check=True)

print("Repository cloned to:", PROJECT_DIR)

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "gdown"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)

print("Dependencies installed.")

In [ ]:
import gdown

if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

gdown.download(id=GDRIVE_FILE_ID, output=str(ARCHIVE_PATH), quiet=False)

with zipfile.ZipFile(ARCHIVE_PATH, "r") as zip_file:
    zip_file.extractall(EXTRACT_DIR)

print("Downloaded archive:", ARCHIVE_PATH)
print("Extracted to:", EXTRACT_DIR)

In [ ]:
def find_first(root: Path, name: str):
    matches = [path for path in root.rglob(name)]
    if not matches:
        raise FileNotFoundError(f"Could not find {name} under {root}")
    return matches[0]

def find_dir(root: Path, dir_name: str):
    matches = [path for path in root.rglob(dir_name) if path.is_dir()]
    if not matches:
        raise FileNotFoundError(f"Could not find directory {dir_name} under {root}")
    return matches[0]

fold_csv_source = find_first(EXTRACT_DIR, "fold-lists.csv")
cdma_features_source = find_dir(EXTRACT_DIR, "cdma_features")

target_data_dir = PROJECT_DIR / "data"
target_data_dir.mkdir(parents=True, exist_ok=True)

target_fold_csv = target_data_dir / "fold-lists.csv"
target_cdma_dir = target_data_dir / "cdma_features"

if target_fold_csv.exists():
    target_fold_csv.unlink()
if target_cdma_dir.exists():
    shutil.rmtree(target_cdma_dir)

shutil.copy2(fold_csv_source, target_fold_csv)
shutil.copytree(cdma_features_source, target_cdma_dir)

print("Prepared Kaggle data folder at:", target_data_dir)
print("fold-lists.csv:", target_fold_csv.exists())
print("cdma_features exists:", target_cdma_dir.exists())

In [ ]:
import numpy as np

rt_dir = PROJECT_DIR / "data" / "cdma_features" / "rt"
it_dir = PROJECT_DIR / "data" / "cdma_features" / "it"

rt_files = sorted(rt_dir.glob("*_frames.npy"))
it_files = sorted(it_dir.glob("*_frames.npy"))

print("RT files:", len(rt_files))
print("IT files:", len(it_files))

sample_rt = np.load(rt_dir / "01_CF56_1_frames.npy")
sample_it = np.load(it_dir / "01_CF56_1_frames.npy")
print("Sample RT shape:", sample_rt.shape)
print("Sample IT shape:", sample_it.shape)

In [ ]:
for rep in REPS:
    for mode in CONDITIONS:
        command = [
            "python",
            "run_module4.py",
            "--mode", mode,
            "--all-folds",
            "--rep", str(rep),
            "--epochs", str(EPOCHS),
        ]
        print("Running:", " ".join(command))
        subprocess.run(command, cwd=PROJECT_DIR, check=True)

print("All Module 4 runs completed.")

In [ ]:
import pandas as pd

results_dir = PROJECT_DIR / "results" / "module4"
rows = []

for rep in REPS:
    for mode in CONDITIONS:
        report_path = results_dir / f"{mode}_rep{rep}_all_folds_report.txt"
        if not report_path.exists():
            print("Missing report:", report_path)
            continue

        metrics = {}
        for line in report_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line.startswith("- accuracy:"):
                metrics["accuracy"] = float(line.split(":", 1)[1].strip())
            elif line.startswith("- precision:"):
                metrics["precision"] = float(line.split(":", 1)[1].strip())
            elif line.startswith("- recall:"):
                metrics["recall"] = float(line.split(":", 1)[1].strip())
            elif line.startswith("- f1:"):
                metrics["f1"] = float(line.split(":", 1)[1].strip())

        rows.append({
            "rep": rep,
            "mode": mode,
            "accuracy": metrics.get("accuracy"),
            "precision": metrics.get("precision"),
            "recall": metrics.get("recall"),
            "f1": metrics.get("f1"),
            "report_path": str(report_path),
        })

summary_df = pd.DataFrame(rows).sort_values(["rep", "mode"]).reset_index(drop=True)
display(summary_df)

summary_csv = results_dir / "module4_kaggle_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print("Saved summary CSV:", summary_csv)

## Notes
- Default setup runs 1 repetition. Change `REPS` to `list(range(1, 11))` for 10 repetitions.
- Default setup runs full 300 epochs.
- During debugging, run a smaller subset of conditions first, then expand to all 13.
- Results are written under `results/module4` inside the cloned repository.
- You can download outputs from Kaggle after run completion.